In [5]:
!pip install -q keras
!pip install -q keras_tqdm
!pip install -q librosa
!pip install -q keras-tuner

In [1]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# Data & Audio settings
SR = 22050
N_MELS = 128
FMAX = 8000
FIXED_TIME_STEPS = 150
NUM_CLASSES = 10

# Training parameters
BATCH_SIZE = 32
EPOCHS = 50
N_FOLDS = 5
SEED = 42

# Paths
TRAIN_CSV_PATH = "dataset/Train.csv"
TRAIN_AUDIO_DIR = "dataset/Train"
TEST_CSV_PATH = "dataset/Test_Public.csv"
TEST_AUDIO_DIR = "dataset/Test_Public"

# Set random seed
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)



#  Load and Inspect Metadata

In [2]:
df = pd.read_csv(TRAIN_CSV_PATH)
df["filepath"] = df["file_name"].apply(lambda x: os.path.join(TRAIN_AUDIO_DIR, x))

file_paths = df["filepath"].values
labels = df["classID"].values

# Optional: check distribution
for cls, count in pd.Series(labels).value_counts().sort_index().items():
    print(f"Class {cls}: {count} samples")


Class 0: 800 samples
Class 1: 368 samples
Class 2: 800 samples
Class 3: 800 samples
Class 4: 800 samples
Class 5: 801 samples
Class 6: 291 samples
Class 7: 828 samples
Class 8: 769 samples
Class 9: 800 samples


# Augmentation

In [3]:
def augment_spectrogram(spectrogram):
    # Time stretch (resize width)
    time_stretch = random.uniform(0.8, 1.2)
    new_width = int(spectrogram.shape[1] * time_stretch)
    stretched = tf.image.resize(spectrogram, [spectrogram.shape[0], new_width])
    stretched = tf.image.resize(stretched, [spectrogram.shape[0], spectrogram.shape[1]])

    # Frequency masking (in NumPy)
    mask = stretched.numpy()
    freq_mask_param = int(spectrogram.shape[0] * 0.15)
    f0 = random.randint(0, spectrogram.shape[0] - freq_mask_param)
    mask[f0:f0 + freq_mask_param, :, :] = 0

    # Back to Tensor + Add noise
    masked = tf.convert_to_tensor(mask, dtype=tf.float32)
    noise = tf.random.normal(shape=tf.shape(masked), mean=0.0, stddev=0.01)
    augmented = masked + noise

    # Normalize
    min_val = tf.reduce_min(augmented)
    max_val = tf.reduce_max(augmented)
    augmented = (augmented - min_val) / (max_val - min_val + 1e-6)

    return augmented


# Dataset  -- Creating Augmented Dataset

In [4]:
def create_augmented_dataset(file_paths, labels, batch_size=32, augment=True):
    def load_and_process(file_path, label):
        path = file_path.numpy().decode("utf-8")
        try:
            y, sr = librosa.load(path, sr=SR)
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX)
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

            mel_db = np.expand_dims(mel_db, axis=-1)
            mel_db = tf.image.resize(mel_db, (128, FIXED_TIME_STEPS))

            if augment and random.random() < 0.7:
                mel_db = augment_spectrogram(mel_db)

        except Exception as e:
            print(f"Error loading {path}: {e}")
            mel_db = tf.zeros((128, FIXED_TIME_STEPS, 1))

        mel_db = tf.reshape(mel_db, (128, FIXED_TIME_STEPS, 1))
        return mel_db, tf.cast(label, tf.int32)

    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    dataset = dataset.map(lambda x, y: tf.py_function(
        load_and_process, inp=[x, y], Tout=[tf.float32, tf.int32]
    ), num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.map(
        lambda x, y: (tf.ensure_shape(x, [128, FIXED_TIME_STEPS, 1]), tf.ensure_shape(y, []))
    )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Model

In [6]:
def build_model(hp):
    lr = hp.Float("learning_rate", 1e-4, 1e-3, sampling="log")
    dropout = hp.Float("dropout", 0.3, 0.6, step=0.1)
    l2_val = hp.Float("l2", 1e-5, 1e-3, sampling="log")
    
    inputs = tf.keras.layers.Input(shape=(128, FIXED_TIME_STEPS, 1))

    # Initial block (unchanged)
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 1
    residual = x
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 2
    residual = layers.Conv2D(64, (1, 1), padding='same')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 3
    residual = layers.Conv2D(128, (1, 1), padding='same')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Final Dense Head (only dropout and l2 are tunable)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_val))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Tuning Hyperparameter -- path issus unsolved

In [8]:
# === Tuner Setup ===
import keras_tuner as kt
from sklearn.model_selection import train_test_split

TUNER_DIR = "tuner_results"

# Create train/val split just for tuner search
tune_train_paths, tune_val_paths, tune_train_labels, tune_val_labels = train_test_split(
    file_paths,
    labels,
    test_size=0.2,      # Using 80% of the data
    stratify=labels,
    random_state=SEED
)

# Create tf.data.Dataset objects for tuner
tune_train_ds = create_augmented_dataset(tune_train_paths, tune_train_labels, BATCH_SIZE, augment=True)
tune_val_ds = create_augmented_dataset(tune_val_paths, tune_val_labels, BATCH_SIZE, augment=False)

# Define tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=13,      
    factor=3,  # Aggressiveness of early stopping
    hyperband_iterations=1,   # try 2 later ###
    directory=TUNER_DIR, #####  save path
    project_name='mel_cnn_finetune',
    overwrite=False
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,  # More aggressive for tuning phase
    restore_best_weights=True
)

tuner.search(
    tune_train_ds,
    validation_data=tune_val_ds,
    epochs=15,  # Max epochs
    callbacks=[early_stopping],
    verbose=1
)

# Save best HPs
best_hp = tuner.get_best_hyperparameters(1)[0]
print("\nBest Hyperparameters Found:")
for param in best_hp.values:
    print(f"{param}: {best_hp.get(param)}")


Trial 17 Complete [00h 45m 31s]
val_accuracy: 0.801699697971344

Best val_accuracy So Far: 0.801699697971344
Total elapsed time: 02h 53m 28s

Search: Running Trial #18

Value             |Best Value So Far |Hyperparameter
0.00037253        |0.00032226        |learning_rate
0.5               |0.3               |dropout
0.00044891        |1.2605e-05        |l2
13                |13                |tuner/epochs
5                 |5                 |tuner/initial_epoch
2                 |2                 |tuner/bracket
2                 |2                 |tuner/round
0015              |0013              |tuner/trial_id

Epoch 6/13
177/177 [==============================] - 416s 2s/step - loss: 0.5310 - accuracy: 0.8473 - val_loss: 0.9457 - val_accuracy: 0.7011
Epoch 7/13
177/177 [==============================] - 264s 1s/step - loss: 0.4569 - accuracy: 0.8717 - val_loss: 1.1717 - val_accuracy: 0.6544
Epoch 8/13
172/177 [============================>.] - ETA: 6s - loss: 0.4127 - accuracy:

KeyboardInterrupt: 

In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)


# Train  --- possible path issues

In [ ]:
# === K-Fold Training Loop ===

MODEL_DIR = os.path.join(os.getcwd(), "model")
os.makedirs(MODEL_DIR, exist_ok=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(file_paths, labels)):
    print(f"\n Starting Fold {fold + 1}/{N_FOLDS}")

    # Split data for this fold
    train_files = file_paths[train_idx]
    val_files = file_paths[val_idx]
    train_labels_fold = labels[train_idx]
    val_labels_fold = labels[val_idx]

    # Compute class weights
    class_weights = compute_class_weight('balanced', classes=np.unique(train_labels_fold), y=train_labels_fold)
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

    # Create datasets
    train_ds = create_augmented_dataset(train_files, train_labels_fold, BATCH_SIZE, augment=True)
    val_ds = create_augmented_dataset(val_files, val_labels_fold, BATCH_SIZE, augment=False)

    # Build model with best hyperparams
    model = build_model(best_hp)

    # Define callbacks
    model_path = f"model/fold_{fold + 1}_best_model.keras"    # model save path  model\kl
    callbacks = [
        EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_accuracy', patience=5, factor=0.5, min_lr=1e-5, verbose=1),
        ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, verbose=1)
    ]

    # Train
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )

    # Record best fold metrics
    best_val_acc = max(history.history['val_accuracy'])
    best_val_loss = min(history.history['val_loss'])

    print(f" Fold {fold + 1}: Best Val Accuracy = {best_val_acc:.4f}, Loss = {best_val_loss:.4f}")
    fold_results.append({
        "fold": fold + 1,
        "val_accuracy": best_val_acc,
        "val_loss": best_val_loss,
        "model_path": model_path
    })


### Test


In [19]:
# === Cell 8: Evaluate Fold Models on Test_Public and Compare ===

from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import load_model

# Load test data
test_df = pd.read_csv(TEST_CSV_PATH)
test_df["filepath"] = test_df["file_name"].apply(lambda x: os.path.join(TEST_AUDIO_DIR, x))
test_file_paths = test_df["filepath"].values
test_labels = test_df["classID"].values

# Create test dataset
test_dataset = create_augmented_dataset(test_file_paths, test_labels, BATCH_SIZE, augment=False)

# Track test set performance
print("\n Evaluating saved fold models on Test_Public...\n")
for fold_info in fold_results:
    fold_num = fold_info["fold"]
    model_path = fold_info["model_path"]

    print(f" Fold {fold_num} Evaluation:")

    # Load model
    model = load_model(model_path)

    # Predict on full test set
    all_preds, all_labels = [], []
    for batch_x, batch_y in test_dataset:
        logits = model.predict(batch_x, verbose=0)
        preds = np.argmax(logits, axis=1)

        all_preds.extend(preds)
        all_labels.extend(batch_y.numpy())

    # Accuracy & Report
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, digits=4)

    print(f" Accuracy: {acc:.4f}")
    print(report)
    print("-" * 50)

    # Store back
    fold_info["test_accuracy"] = acc
    fold_info["classification_report"] = classification_report(
        all_labels, all_preds, digits=4, output_dict=True
    )

# Summary table
print("\n Fold Comparison on Test Set:")
for f in fold_results:
    print(f"Fold {f['fold']}: Test Accuracy = {f['test_accuracy']:.4f}")


 Evaluating saved models on Test_Public...

 Fold 1 Evaluation:
 Accuracy: 0.7683
 Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.2000    0.3333        10
           1     1.0000    1.0000    1.0000         3
           2     0.9091    1.0000    0.9524        10
           3     1.0000    0.8000    0.8889        10
           4     0.5333    0.8000    0.6400        10
           5     0.5000    0.3333    0.4000         9
           6     1.0000    1.0000    1.0000         4
           7     0.5714    1.0000    0.7273         8
           8     1.0000    0.8750    0.9333         8
           9     0.8333    1.0000    0.9091        10

    accuracy                         0.7683        82
   macro avg     0.8347    0.8008    0.7784        82
weighted avg     0.8150    0.7683    0.7454        82

--------------------------------------------------
 Fold 2 Evaluation:
 Accuracy: 0.7439
 Classification Report:
              precisio

### Detailed Tests


### Test With Google Drive

In [ ]:
"""
Tarık Buğra Ay - 042101100
"""

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import librosa
import gdown
from tqdm import tqdm

# ===================== CONFIG =====================
MODEL_URL = "https://drive.google.com/uc?export=download&id=1bbGUu1q4-Q8lGD3axKNtrDx1I6qxxras"   #Google Drive
TEST_CSV_PATH = "/kaggle/input/yldz-teknik-proje-1-train-dataset/Test_Public.csv"                #Labels
TEST_AUDIO_DIR = "/kaggle/input/yldz-teknik-proje-1-train-dataset/Test_Public/"                  #Audios

SAMPLE_RATE = 22050
N_MELS = 128
FMAX = 8000
FIXED_TIME_STEPS = 150
BATCH_SIZE = 32
# ===================================================

# Download model from Google Drive
print("Downloading model...")
model_path = gdown.download(MODEL_URL, quiet=False)
model = tf.keras.models.load_model(model_path)
print("Model loaded.\n")

# Load test CSV
df = pd.read_csv(TEST_CSV_PATH)
file_paths = [os.path.join(TEST_AUDIO_DIR, fname) for fname in df["file_name"]]
true_labels = df["classID"].values

# Audio preprocessing function
def load_and_process(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    n_fft = min(2048, len(y) // 2)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX, n_fft=n_fft)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())
    mel_db = tf.image.resize(mel_db[..., np.newaxis], (128, FIXED_TIME_STEPS))
    return mel_db.numpy()

# Create dataset
print("Preprocessing test set...")
processed_data = []
for path in tqdm(file_paths):
    mel = load_and_process(path)
    processed_data.append(mel)

X_test = np.stack(processed_data, axis=0)
y_test = np.array(true_labels)

# Predict in batch
print("\nPredicting...")
pred_probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
pred_classes = np.argmax(pred_probs, axis=1)

# Accuracy
accuracy = np.mean(pred_classes == y_test) * 100
print(f"\nAccuracy: {accuracy:.2f}%")

#======================================= NOT============================================================================
# Hocam, bu kısmın sonuçlarını benimle paylaşırsanız çok sevinirim. 
# Böylece modelin nerelerde zayıf kaldığını görüp, diğer modeli daha verimli şekilde geliştirebilirim. Teşekkür ederim.
#=======================================================================================================================

# === Classification Report ===
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


report = classification_report(y_test, pred_classes, output_dict=True)
report_df = pd.DataFrame(report).transpose()
print("\nClassification Report:")
print(report_df)
